In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from scipy.stats import norm
import utilities as utils

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GaussianNetwork(nn.Module):
    """
    Neural network mapping x -> parameters of an N_y-dimensional Gaussian.

    Outputs
    -------
    mean : Tensor, shape (..., N_y)
        Gaussian mean.

    scale_tril : Tensor, shape (..., N_y, N_y)
        Lower-triangular Cholesky factor L such that
            covariance = L @ L.T

    covariance : Tensor, shape (..., N_y, N_y)
        Gaussian covariance matrix.
    """

    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dims=(128, 128),
        min_std=1e-4,
    ):
        super().__init__()

        self.output_dim = output_dim
        self.min_std = min_std

        # Shared feature-processing network
        layers = []
        d = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(d, hidden_dim),
                nn.ReLU(),
            ])
            d = hidden_dim

        self.backbone = nn.Sequential(*layers)

        # Mean requires N_y parameters
        self.mean_head = nn.Linear(d, output_dim)

        # Lower triangular matrix requires
        # N_y * (N_y + 1) / 2 parameters
        n_tril = output_dim * (output_dim + 1) // 2
        self.cholesky_head = nn.Linear(d, n_tril)

        # Indices of lower-triangular matrix elements
        tril_indices = torch.tril_indices(
            row=output_dim,
            col=output_dim,
            offset=0,
        )

        self.register_buffer("tril_indices", tril_indices)


    def forward(self, x):
        h = self.backbone(x)

        # Mean
        mean = self.mean_head(h)

        # Raw parameters for Cholesky factor
        raw_tril = self.cholesky_head(h)

        batch_shape = x.shape[:-1]

        L = torch.zeros(
            *batch_shape,
            self.output_dim,
            self.output_dim,
            device=x.device,
            dtype=x.dtype,
        )

        L[..., self.tril_indices[0], self.tril_indices[1]] = raw_tril

        # Diagonal must be positive.
        diag_idx = torch.arange(self.output_dim, device=x.device)

        raw_diag = L[..., diag_idx, diag_idx]

        L[..., diag_idx, diag_idx] = (
            F.softplus(raw_diag) + self.min_std
        )

        covariance = L @ L.transpose(-1, -2)

        return mean, L, covariance

In [3]:
def generate_trials(n_trials, mirror_trials=False, balanced_starts=False, seed=None):
    """Generate trials from a two-state switching Gaussian process.

    Parameters
    ----------
    n_trials : int
        Number of independent trials to generate.
    balanced_starts : bool, default=False
        If True, exactly half of the trials start in each source
        distribution. This requires an even ``n_trials``.
    seed : int or None, default=None
        Seed for NumPy's random-number generator. Use an integer for
        reproducible simulations.

    Returns
    -------
    X : ndarray of int, shape (n_trials, 12)
        Clipped and rounded observations for each trial.
    Y : ndarray of int, shape (n_trials,)
        Identity of the source for the final draw: 0 denotes the Gaussian
        with mean -17 and 1 denotes the Gaussian with mean 17.
    """
    if isinstance(n_trials, (bool, np.bool_)) or not isinstance(n_trials, (int, np.integer)):
        raise TypeError("n_trials must be an integer")
    if n_trials < 0:
        raise ValueError("n_trials must be non-negative")
    if (balanced_starts or mirror_trials) and n_trials % 2:
        raise ValueError("balanced_starts=True or mirror_trials=True requires an even n_trials")

    rng = np.random.default_rng(seed)
    n_draws = 12
    source_means = np.array([-17.0, 17.0])

    if mirror_trials:
        n_trials_ = n_trials // 2
    else:
        n_trials_ = n_trials

    if balanced_starts:
        starts = np.repeat([0, 1], n_trials_ // 2)
        rng.shuffle(starts)
    else:
        starts = rng.integers(0, 2, size=n_trials_)

    # switches[:, j] indicates whether the source changes between draws
    # j and j + 1. Cumulative parity therefore gives the source at each draw.
    switches = rng.random((n_trials_, n_draws - 1)) < 0.08
    source_paths = np.column_stack(
        [starts, starts[:, None] ^ np.logical_xor.accumulate(switches, axis=1)]
    ).astype(int)

    raw_draws = rng.normal(loc=source_means[source_paths], scale=29.0)
    X = np.rint(np.clip(raw_draws, -90, 90)).astype(int)
    Y = source_paths[:, -1].copy()
    if mirror_trials:
        X = np.concatenate([X, -X], axis=0)
        Y = np.concatenate([Y, 1-Y], axis=0)
    return X, Y

In [4]:
Xsample,Ysample = generate_trials(100000, mirror_trials=True, balanced_starts=True, seed=234)

In [52]:
test_enc = GaussianNetwork(input_dim=12, output_dim=2, hidden_dims=(128, 128), min_std=1e-4)
test_enc.forward(torch.from_numpy(Xsample[:10]).float())[2]

tensor([[[ 1.3057e+00, -5.1041e-01],
         [-5.1041e-01,  7.3727e-01]],

        [[ 1.2359e+01, -9.2911e+00],
         [-9.2911e+00,  8.2029e+00]],

        [[ 2.2363e-02,  1.1757e-02],
         [ 1.1757e-02,  9.6068e-02]],

        [[ 7.4850e+00, -1.4447e+00],
         [-1.4447e+00,  7.1893e+00]],

        [[ 7.0954e+00,  9.3081e-02],
         [ 9.3081e-02,  1.0836e+01]],

        [[ 1.1474e+00, -1.9437e+00],
         [-1.9437e+00,  5.2947e+00]],

        [[ 7.8731e-02, -1.2202e+00],
         [-1.2202e+00,  2.4244e+01]],

        [[ 4.5224e+00, -2.8640e+00],
         [-2.8640e+00,  3.7007e+00]],

        [[ 6.9979e-01, -7.5487e-01],
         [-7.5487e-01,  9.2517e-01]],

        [[ 1.5942e-02,  1.9157e-01],
         [ 1.9157e-01,  2.3923e+00]]], grad_fn=<UnsafeViewBackward0>)

In [86]:
def gaussian_kl_divergence_torch(
    true_mean,
    true_scale_tril,
    approx_mean,
    approx_scale_tril,
):
    true_dist = torch.distributions.MultivariateNormal(
        loc=true_mean,
        scale_tril=true_scale_tril,
    )

    approx_dist = torch.distributions.MultivariateNormal(
        loc=approx_mean,
        scale_tril=approx_scale_tril,
    )

    return torch.distributions.kl_divergence(
        true_dist,
        approx_dist,
    )

def p_RgX_mc_est(mean,L,n_samples=1000):
    """Monte Carlo estimate of p(R|X) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_ZgX = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

    ZgX_samples = p_ZgX.rsample((n_samples,))
    RgX_samples = torch.argmax(ZgX_samples, dim=-1)
    RgX_onehot = F.one_hot(RgX_samples, num_classes=mean.shape[-1])
    p_RgX = RgX_onehot.float().mean(dim=0)
    return p_RgX

def p_RgY_mc_est(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of p(R|Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgX = p_RgX_mc_est(mean,L,n_samples=n_samples)
    Y_onehot = F.one_hot(Y, num_classes=mean.shape[-1])
    p_RgY = p_RgX.T @ Y_onehot.float() / Y_onehot.float().sum(dim=0, keepdim=True)

    return p_RgY

def I_RY(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of I(R;Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgY = p_RgY_mc_est(mean,L,Y,n_samples=n_samples)
    p_Y = F.one_hot(Y, num_classes=mean.shape[-1]).float().mean(dim=0,keepdim=True)
    p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)
    I_RY = (p_RgY * p_Y * (p_RgY / p_R).log()).sum()
    return I_RY

def variational_obj(beta,mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of the variational objective for a Gaussian encoder.

    Parameters
    ----------
    beta : float
        Trade-off parameter between I(X;R) and I(R;Y).

    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    DKL_ZgX = gaussian_kl_divergence_torch(
        true_mean=mean,
        true_scale_tril=L,
        approx_mean=torch.zeros(mean.shape[-1],device=mean.device, dtype=mean.dtype),   # torch.zeros_like(mean, device=mean.device, dtype=mean.dtype),
        approx_scale_tril=torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype)     # torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype).expand(mean.shape[0], -1, -1),
    )

    I_XR_est = DKL_ZgX.mean()
    I_RY_est = I_RY(mean,L,Y,n_samples=n_samples)
    return I_XR_est - beta * I_RY_est

In [74]:
mean = test_enc.forward(torch.from_numpy(Xsample).float())[0]
L = test_enc.forward(torch.from_numpy(Xsample).float())[1]
Y = torch.from_numpy(Ysample).long()
beta = 5.0

In [77]:
DKL_ZgX = gaussian_kl_divergence_torch(
        true_mean=mean,
        true_scale_tril=L,
        approx_mean=torch.zeros_like(mean, device=mean.device, dtype=mean.dtype),
        approx_scale_tril=torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype).expand(mean.shape[0], -1, -1),
    )

In [78]:
DKL_ZgX

tensor([ 1.7628, 15.1412, 18.1263,  ...,  4.5185, 17.1012, 22.2232],
       grad_fn=<AddBackward0>)

In [87]:
variational_obj(beta,mean,L,Y,n_samples=1000)

tensor(12.2654, grad_fn=<SubBackward0>)

In [88]:
variational_obj(1.0,mean,L,Y,n_samples=1000)

tensor(12.3424, grad_fn=<SubBackward0>)

In [8]:
mean = test_enc.forward(torch.from_numpy(Xsample[:10]).float())[0]
L = test_enc.forward(torch.from_numpy(Xsample[:10]).float())[1]

In [9]:
gaussian_kl_divergence_torch(
    true_mean=mean,
    true_scale_tril=L,
    approx_mean=torch.zeros_like(mean, device=mean.device, dtype=mean.dtype),
    approx_scale_tril=torch.eye(mean.shape[-1], device=mean.device, dtype=mean.dtype).expand(mean.shape[0], -1, -1),
)

tensor([10.0329, 13.8889,  9.0007,  5.3260,  9.9812, 17.7749, 40.7789, 36.2143,
         3.5432,  3.5954], grad_fn=<AddBackward0>)

In [53]:
mean = test_enc.forward(torch.from_numpy(Xsample[:10000]).float())[0]
L = test_enc.forward(torch.from_numpy(Xsample[:10000]).float())[1]
Y = torch.from_numpy(Ysample[:10000]).long()

In [63]:
p_RgY = p_RgY_mc_est(mean,L,Y,n_samples=1000)
p_RgY

tensor([[0.5260, 0.3428],
        [0.4740, 0.6572]])

In [64]:
p_Y = F.one_hot(Y, num_classes=mean.shape[-1]).float().mean(dim=0,keepdim=True)
p_Y

tensor([[0.4962, 0.5038]])

In [65]:
p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)
p_R

tensor([[0.4337],
        [0.5663]])

In [68]:
(p_RgY * p_Y * (p_RgY / p_R).log()).sum(dim=1, keepdim=True).sum()

tensor(0.0172)

In [70]:
(p_RgY * p_Y * (p_RgY / p_R).log()).sum()

tensor(0.0172)

In [69]:
I_RY(mean,L,Y,n_samples=1000)

tensor(0.0171)

In [ ]:
def I_RY(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of I(R;Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgY = p_RgY_mc_est(mean,L,Y,n_samples=n_samples)
    p_Y = F.one_hot(Y, num_classes=mean.shape[-1]).float().mean(dim=0,keepdim=True)
    p_R = (p_RgY * p_Y).sum(dim=1, keepdim=True)
    I_RY = (p_RgY * p_Y * (p_RgY / p_R).log()).sum()
    return I_RY

In [39]:
p_RgX = p_RgX_mc_est(mean,L,n_samples=1000)
p_RgX.shape

torch.Size([10, 2])

In [40]:
p_RgX

tensor([[0.0830, 0.9170],
        [0.2530, 0.7470],
        [0.5740, 0.4260],
        [0.3660, 0.6340],
        [0.5270, 0.4730],
        [0.7120, 0.2880],
        [0.4610, 0.5390],
        [0.7260, 0.2740],
        [0.0370, 0.9630],
        [0.1880, 0.8120]])

In [42]:
Y_onehot = F.one_hot(torch.from_numpy(Ysample[:10]), num_classes=mean.shape[-1])
Y_onehot.shape

torch.Size([10, 2])

In [45]:
p_RgX.T @ Y_onehot.float()

tensor([[1.1150, 2.8120],
        [1.8850, 4.1880]])

In [48]:
p_RgX.T @ Y_onehot.float() / Y_onehot.float().sum(dim=0, keepdim=True)

tensor([[0.3717, 0.4017],
        [0.6283, 0.5983]])

In [46]:
p_RgX.T

tensor([[0.0830, 0.2530, 0.5740, 0.3660, 0.5270, 0.7120, 0.4610, 0.7260, 0.0370,
         0.1880],
        [0.9170, 0.7470, 0.4260, 0.6340, 0.4730, 0.2880, 0.5390, 0.2740, 0.9630,
         0.8120]])

In [43]:
Y_onehot

tensor([[0, 1],
        [0, 1],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [0, 1],
        [1, 0],
        [0, 1]])

In [47]:
Y_onehot.float().sum(dim=0, keepdim=True)

tensor([[3., 7.]])

In [ ]:
def p_RgY_mc_est(mean,L,Y,n_samples=1000):
    """Monte Carlo estimate of p(R|Y) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    Y : Tensor, shape (n_trials,)
        Ground-truth source identity for each trial.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_RgX = p_RgX_mc_est(mean,L,n_samples=n_samples)
    Y_onehot = F.one_hot(Y, num_classes=mean.shape[-1])
    p_RgY = p_RgX.T @ Y_onehot.float() / Y_onehot.float().sum(dim=0, keepdim=True)

    return p_RgY

In [13]:
p_ZgX = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

In [16]:
ZgX_samples = p_ZgX.rsample((1000,))
ZgX_samples.shape

torch.Size([1000, 10, 2])

In [32]:
ZgX_samples[2]

tensor([[ 1.3138,  3.6046],
        [-1.4387,  7.3054],
        [ 1.1912,  1.5843],
        [-1.6487, -2.1961],
        [ 0.7811,  3.2515],
        [ 2.7614,  3.3366],
        [-2.8398, -6.3952],
        [ 3.2114, -6.5034],
        [-2.9531,  2.6532],
        [-2.0501,  3.0222]], grad_fn=<SelectBackward0>)

In [18]:
R_samples = torch.argmax(ZgX_samples, dim=-1)
R_samples.shape

torch.Size([1000, 10])

In [22]:
R_onehot = F.one_hot(R_samples, num_classes=mean.shape[-1])
R_onehot.shape

torch.Size([1000, 10, 2])

In [33]:
R_samples[2]

tensor([1, 1, 1, 0, 1, 1, 0, 0, 1, 1])

In [37]:
R_onehot.float().mean(dim=0)

tensor([[0.0820, 0.9180],
        [0.2740, 0.7260],
        [0.6030, 0.3970],
        [0.3610, 0.6390],
        [0.4920, 0.5080],
        [0.6930, 0.3070],
        [0.4590, 0.5410],
        [0.7290, 0.2710],
        [0.0430, 0.9570],
        [0.1790, 0.8210]])

In [23]:
R_onehot

tensor([[[0, 1],
         [1, 0],
         [1, 0],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        [[1, 0],
         [0, 1],
         [1, 0],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        [[0, 1],
         [0, 1],
         [0, 1],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        ...,

        [[0, 1],
         [0, 1],
         [1, 0],
         ...,
         [1, 0],
         [0, 1],
         [0, 1]],

        [[0, 1],
         [0, 1],
         [0, 1],
         ...,
         [1, 0],
         [0, 1],
         [1, 0]],

        [[0, 1],
         [0, 1],
         [0, 1],
         ...,
         [1, 0],
         [0, 1],
         [1, 0]]])

In [19]:
R_samples

tensor([[1, 0, 0,  ..., 0, 1, 1],
        [0, 1, 0,  ..., 0, 1, 1],
        [1, 1, 1,  ..., 0, 1, 1],
        ...,
        [1, 1, 0,  ..., 0, 1, 1],
        [1, 1, 1,  ..., 0, 1, 0],
        [1, 1, 1,  ..., 0, 1, 0]])

In [12]:
p_RgX_mc_est(mean,L,n_samples=1000)

TypeError: argmax(): argument 'dim' must be int, not tuple

In [ ]:
def p_RgX_mc_est(mean,L,n_samples=1000):
    """Monte Carlo estimate of p(R|X) for a Gaussian encoder.

    Parameters
    ----------
    mean : Tensor, shape (n_trials, N_y)
        Mean of the Gaussian encoder.
    L : Tensor, shape (n_trials, N_y, N_y)
        Lower-triangular Cholesky factor of the covariance of the Gaussian
        encoder.
    n_samples : int, default=1000
        Number of Monte Carlo samples to use for the estimate."""

    p_ZgX = torch.distributions.MultivariateNormal(
        loc=mean,
        scale_tril=L,
    )

    ZgX_samples = p_ZgX.rsample((n_samples,))
    RgX_samples = torch.argmax(ZgX_samples, dim=-1)
    RgX_onehot = F.one_hot(RgX_samples, num_classes=mean.shape[-1])
    p_RgX = RgX_onehot.float().mean(dim=-2)
    return p_RgX